### PACOTES

In [3]:
import pandas as pd
import numpy as np
import itertools
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.spatial.distance import euclidean
from sklearn.preprocessing import RobustScaler

import time


In [4]:

inicio = time.time()
df_resumo = pd.read_csv('resumo_features.csv')
df = pd.read_csv('creditcard.csv')

# AJUSTE DE TIPOS
df_resumo['Comparacao_dis_whith'] = pd.to_numeric(
    df_resumo['Comparacao_dis_whith'],
    errors='coerce'
)

# MANN-WHITNEY SCORE
# Quanto menor o p-value -> melhor
df_resumo['MannWhitney_Score'] = -np.log10(
    df_resumo['Comparacao_dis_whith'] + 1e-300
)

# CORRELAÇÃO ABSOLUTA
df_resumo['Corr_spearman_abs'] = abs(
    df_resumo['Correlacao_spearman']
)

# AJUSTE DE MÉTRICAS DISTRIBUCIONAIS

# KL e Wasser podem explodir
# Aplicamos log para estabilizar

df_resumo['divergencia_kl'] = np.log1p(
    abs(df_resumo['divergencia_kl'])
)

df_resumo['Wasser'] = np.log1p(
    abs(df_resumo['Wasser'])
)

# AJUSTE DE NORMALIDADE

# Quanto MENOR distância da normalidade -> melhor
df_resumo['Normalidade_Score'] = 1 / (
    1 + abs(df_resumo['Normalidade'])
)

# AJUSTE DE ASSIMETRIA
df_resumo['Skewness_Score'] = 1 / (
    1 + abs(df_resumo['Assimetria'])
)


# AJUSTE DE CURTOSE
df_resumo['Kurtosis_Score'] = 1 / (
    1 + abs(df_resumo['Curtose'])
)

# AJUSTE DE OUTLIERS
df_resumo['Outlier_Score'] = 1 / (
    1 + df_resumo['Qnd_Outliers']
)

# MÉTRICAS UTILIZADAS
metricas = [

    # INDISPENSÁVEL
    'Curva ROC',
    'KS',

    # MUITO IMPORTANTE
    'divergencia_kl',
    'Wasser',

    # IMPORTANTE
    'inf_mutua',
    'MannWhitney_Score',

    # BOM
    'Normalidade_Score',
    'Sep_mediana_norm',

    # ÚTIL
    'Skewness_Score',
    'Kurtosis_Score',

    # ACESSÓRIO
    'Corr_spearman_abs',
    'Outlier_Score'
]


# TRATAMENTO DE NaN
df_resumo[metricas] = df_resumo[
    metricas
].fillna(0)


# NORMALIZAÇÃO ROBUSTA
# Melhor para fraude/outliers

scaler = RobustScaler()

df_norm = df_resumo.copy()

df_norm[metricas] = scaler.fit_transform(
    df_norm[metricas]
)


# MIN-MAX FINAL
# Após robust scaling

for col in metricas:

    minimo = df_norm[col].min()
    maximo = df_norm[col].max()

    df_norm[col] = (
        (df_norm[col] - minimo)
        /
        (maximo - minimo + 1e-9)
    )


# SCORE FINAL
df_norm['Score_Final'] = (

    # INDISPENSÁVEL -> 0.25
    df_norm['Curva ROC'] * 0.125 +
    df_norm['KS'] * 0.125 +

    # MUITO IMPORTANTE -> 0.21
    df_norm['divergencia_kl'] * 0.105 +
    df_norm['Wasser'] * 0.105 +

    # IMPORTANTE -> 0.17
    df_norm['inf_mutua'] * 0.085 +
    df_norm['MannWhitney_Score'] * 0.085 +

    # BOM -> 0.13
    df_norm['Normalidade_Score'] * 0.065 +
    df_norm['Sep_mediana_norm'] * 0.065 +

    # ÚTIL -> 0.09
    df_norm['Skewness_Score'] * 0.045 +
    df_norm['Kurtosis_Score'] * 0.045 +

    # ACESSÓRIO -> 0.05
    df_norm['Corr_spearman_abs'] * 0.025 +
    df_norm['Outlier_Score'] * 0.025
)


# GARANTIR SEM NaN
df_norm['Score_Final'] = df_norm[
    'Score_Final'
].fillna(0)

# RANKING FINAL
ranking = df_norm.sort_values(
    'Score_Final',
    ascending=False
).reset_index(drop=True)

# POSIÇÃO
ranking.insert(
    0,
    'Posicao_Rank',
    ranking.index + 1
)

# EXIBIÇÃO
pd.set_option(
    'display.max_columns',
    None
)

display(ranking)

# SALVAR CSV
ranking.to_csv(
    'individual_score_feature_individuais.csv',
    index=False
)

print('\nCSV salvo com sucesso!')

,Posicao_Rank,Feature,Normalidade,Qnd_Outliers,Sep_mediana_norm,Comparacao_dis_whith,Correlacao_spearman,inf_mutua,divergencia_kl,Curva ROC,KS,Wasser,Assimetria,Curtose,MannWhitney_Score,Corr_spearman_abs,Normalidade_Score,Skewness_Score,Kurtosis_Score,Outlier_Score,Score_Final
0,1,V14,0,14149,1.000000,1.471581e-260,-0.064613,0.984641,1.000000,1.000000,1.000000,0.211410,1.995165,23.879022,1.000000,1.000000,0.0,0.305756,0.046824,0.000045,0.651760
1,2,V12,0,15348,0.796989,8.416027e-247,-0.062870,0.917286,0.840947,0.972118,0.924927,0.201544,2.278389,20.241493,0.946936,0.972125,0.0,0.274060,0.055062,0.000040,0.595970
2,3,V10,0,9496,0.579452,9.611131e-222,-0.059564,0.908347,0.894367,0.919245,0.950893,0.192730,1.187134,31.987656,0.850284,0.919255,0.0,0.441298,0.035000,0.000080,0.579477
3,4,V11,0,780,0.497847,4.910592e-226,0.060143,0.820345,0.725013,0.928507,0.889460,0.157954,0.356504,1.633872,0.866838,0.928514,0.0,0.748947,0.453109,0.001255,0.573065
4,5,V4,0,11148,0.415041,3.625904e-248,0.063045,0.586806,0.805397,0.974920,0.902628,0.173101,0.676289,2.635388,0.952203,0.974924,0.0,0.594410,0.327929,0.000064,0.561109
5,6,V17,0,7420,0.870823,9.219384e-124,-0.044335,1.000000,0.905760,0.675706,0.875582,0.212480,3.844894,94.798034,0.472352,0.675708,0.0,0.165685,0.011212,0.000109,0.517935
6,7,V3,0,3363,0.486416,1.211048e-219,-0.059278,0.583784,0.718940,0.914680,0.822726,0.212220,2.240144,26.619062,0.842183,0.914681,0.0,0.278017,0.042052,0.000272,0.505049
7,8,V16,0,8184,0.580474,1.808172e-156,-0.049936,0.733854,0.745415,0.765280,0.800845,0.165757,1.100960,10.418927,0.598510,0.765281,0.0,0.461906,0.103528,0.000097,0.486999
8,9,V7,0,8948,0.346706,1.464234e-146,-0.048308,0.457887,0.709440,0.739240,0.767815,0.191285,2.553894,405.600275,0.560292,0.739245,0.0,0.248076,0.001663,0.000086,0.421761
9,10,V9,0,8283,0.272512,8.943723e-154,-0.049499,0.498804,0.482208,0.758284,0.660477,0.127076,0.554677,3.731224,0.588117,0.758292,0.0,0.645688,0.251678,0.000095,0.410763



CSV salvo com sucesso!


### TOP 10


In [5]:

# CARREGAR CSV DO RANKING
ranking = pd.read_csv('individual_score_feature_individuais.csv')

ranking = ranking.sort_values(
    by='Score_Final',
    ascending=False
)

top10 = ranking.head(10)

# ATRIBUINDO ÀS VARIÁVEIS
Primeiro_lugar = top10.iloc[0]['Feature']
Segundo_lugar = top10.iloc[1]['Feature']
Terceiro_lugar = top10.iloc[2]['Feature']
Quarto_lugar = top10.iloc[3]['Feature']
Quinto_lugar = top10.iloc[4]['Feature']
Sexto_lugar = top10.iloc[5]['Feature']
Setimo_lugar = top10.iloc[6]['Feature']
Oitavo_lugar = top10.iloc[7]['Feature']
Nono_lugar = top10.iloc[8]['Feature']
Decimo_lugar = top10.iloc[9]['Feature']

# PRINT TOP 10
print('TOP 10 FEATURES:\n')

for i, row in top10.iterrows():

    print(
        f"{row['Posicao_Rank']}º -> "
        f"{row['Feature']}"
    )

# LISTA FINAL
features_top10 = top10['Feature'].tolist()
print('\nLista Top 10:\n')
print(features_top10)

TOP 10 FEATURES:

1º -> V14
2º -> V12
3º -> V10
4º -> V11
5º -> V4
6º -> V17
7º -> V3
8º -> V16
9º -> V7
10º -> V9

Lista Top 10:

['V14', 'V12', 'V10', 'V11', 'V4', 'V17', 'V3', 'V16', 'V7', 'V9']


 ### CRIANDO CSV DE COMBINACOES 2X2 E 3X3 

In [6]:

# ==================================================
# CONFIG
# ==================================================

RANDOM_STATE = 42

# 5% da população
SAMPLE_PERCENT = 0.05

# FEATURES PCA
features_pca = top10['Feature'].tolist()

# ==================================================
# FUNÇÃO PRINCIPAL
# ==================================================

def calcular_metricas(
    combinacoes,
    dimensao='2D'
):

    resultados = []

    for combo in combinacoes:

        try:

            # =========================
            # DADOS
            # =========================

            X = df[list(combo)].values

            # =========================
            # PADRONIZAÇÃO
            # =========================

            scaler = StandardScaler()

            X_scaled = scaler.fit_transform(X)

            # =========================
            # GMM
            # =========================

            gmm = GaussianMixture(

                n_components=2,
                covariance_type='diag',
                random_state=RANDOM_STATE

            )

            labels = gmm.fit_predict(
                X_scaled
            )

            # =========================
            # AMOSTRA 5%
            # =========================

            sample_size = max(

                100,

                int(
                    len(X_scaled)
                    * SAMPLE_PERCENT
                )

            )

            idx = np.random.choice(

                len(X_scaled),

                size=sample_size,

                replace=False

            )

            X_sample = X_scaled[idx]

            labels_sample = labels[idx]

            # =========================
            # 1. SILHOUETTE
            # =========================

            silhouette = silhouette_score(

                X_sample,
                labels_sample

            )

            # =========================
            # 2. DAVIES-BOULDIN
            # =========================

            davies = davies_bouldin_score(

                X_sample,
                labels_sample

            )

            # =========================
            # 3. CALINSKI-HARABASZ
            # =========================

            calinski = calinski_harabasz_score(

                X_sample,
                labels_sample

            )

            # =========================
            # 4. DISTÂNCIA CENTROIDES
            # =========================

            centro_0 = X_sample[
                labels_sample == 0
            ].mean(axis=0)

            centro_1 = X_sample[
                labels_sample == 1
            ].mean(axis=0)

            centroid_dist = euclidean(

                centro_0,
                centro_1

            )

            # =========================
            # 5. OVERLAP GMM
            # =========================

            probs = gmm.predict_proba(
                X_sample
            )

            overlap = np.mean(

                np.min(probs, axis=1)

            )

            overlap_score = 1 - overlap

            # =========================
            # RESULTADOS
            # =========================

            resultado = {

                'Features': ' | '.join(combo),

                'Silhouette': round(
                    float(silhouette), 6
                ),

                'Davies_Bouldin': round(
                    float(davies), 6
                ),

                'Calinski_Harabasz': round(
                    float(calinski), 6
                ),

                'Dist_Centroides': round(
                    float(centroid_dist), 6
                ),

                'Overlap_GMM': round(
                    float(overlap_score), 6
                )
            }

            resultados.append(resultado)

            # =========================
            # PRINT
            # =========================

            print(

                f'[{dimensao}] '

                f'{" | ".join(combo)} '

                f'| SIL: {silhouette:.4f} '

                f'| DB: {davies:.4f} '

                f'| CH: {calinski:.2f} '

                f'| CENT: {centroid_dist:.4f} '

                f'| OVER: {overlap_score:.4f}'
            )

        except Exception as e:

            print(
                f'Erro em {combo}: {e}'
            )

    return pd.DataFrame(resultados)

# ==================================================
# 2x2
# ==================================================

combinacoes_2d = list(
    itertools.combinations(
        features_pca,
        2
    )
)

print(
    f'\nTotal 2x2: '
    f'{len(combinacoes_2d)}'
)

df_2d = calcular_metricas(

    combinacoes_2d,

    dimensao='2D'

)

# CSV 2D
df_2d.to_csv(

    '2X2_visu_metricas.csv',

    index=False

)

print(
    '\nCSV 2x2 salvo com sucesso!'
)

# ==================================================
# 3x3
# ==================================================

combinacoes_3d = list(
    itertools.combinations(
        features_pca,
        3
    )
)

print(
    f'\nTotal 3x3: '
    f'{len(combinacoes_3d)}'
)

df_3d = calcular_metricas(

    combinacoes_3d,

    dimensao='3D'

)

# CSV 3D
df_3d.to_csv(

    '3X3_visu_metricas.csv',

    index=False

)

print(
    '\nCSV 3x3 salvo com sucesso!'
)

# ==================================================
# DISPLAY
# ==================================================

display(df_2d.head())

display(df_3d.head())
# No final do seu script:
fim = time.time()
tempo_total = fim - inicio

# Convertendo para um formato amigável (minutos e segundos)
minutos = int(tempo_total // 60)
segundos = tempo_total % 60

print(f'\n✓ Processo concluído em: {minutos}m {segundos:.2f}s')


Total 2x2: 45
[2D] V14 | V12 | SIL: 0.5462 | DB: 1.7478 | CH: 2741.18 | CENT: 1.7875 | OVER: 0.9410
[2D] V14 | V10 | SIL: 0.6396 | DB: 2.4924 | CH: 994.40 | CENT: 1.7537 | OVER: 0.9714
[2D] V14 | V11 | SIL: 0.4011 | DB: 4.9997 | CH: 367.22 | CENT: 0.6776 | OVER: 0.8525
[2D] V14 | V4 | SIL: 0.4287 | DB: 3.1998 | CH: 973.99 | CENT: 1.0181 | OVER: 0.8488
[2D] V14 | V17 | SIL: 0.6113 | DB: 1.7162 | CH: 1708.67 | CENT: 2.0215 | OVER: 0.9594
[2D] V14 | V3 | SIL: 0.5198 | DB: 2.0594 | CH: 1693.07 | CENT: 1.6945 | OVER: 0.9199
[2D] V14 | V16 | SIL: 0.4588 | DB: 5.0113 | CH: 378.21 | CENT: 0.7069 | OVER: 0.8727
[2D] V14 | V7 | SIL: 0.6479 | DB: 4.8752 | CH: 302.29 | CENT: 0.9614 | OVER: 0.9709
[2D] V14 | V9 | SIL: 0.4231 | DB: 2.8100 | CH: 1256.67 | CENT: 1.0896 | OVER: 0.8584
[2D] V12 | V10 | SIL: 0.4799 | DB: 1.5813 | CH: 3362.18 | CENT: 1.6101 | OVER: 0.8954
[2D] V12 | V11 | SIL: 0.4608 | DB: 1.2466 | CH: 4083.12 | CENT: 2.0427 | OVER: 0.8754
[2D] V12 | V4 | SIL: 0.4084 | DB: 2.0108 | CH: 2

,Features,Silhouette,Davies_Bouldin,Calinski_Harabasz,Dist_Centroides,Overlap_GMM
0,V14 | V12,0.546199,1.747836,2741.178112,1.787456,0.941045
1,V14 | V10,0.639580,2.492396,994.400390,1.753721,0.971431
2,V14 | V11,0.401122,4.999709,367.216583,0.677617,0.852484
3,V14 | V4,0.428688,3.199823,973.994260,1.018098,0.848775
4,V14 | V17,0.611298,1.716242,1708.672676,2.021481,0.959384


,Features,Silhouette,Davies_Bouldin,Calinski_Harabasz,Dist_Centroides,Overlap_GMM
0,V14 | V12 | V10,0.455383,2.435582,1629.820319,1.522208,0.937417
1,V14 | V12 | V11,0.442753,2.135009,1879.522273,1.781022,0.936897
2,V14 | V12 | V4,0.410957,2.385204,1794.318489,1.544737,0.910421
3,V14 | V12 | V17,0.475969,2.094408,1970.406839,1.761508,0.941782
4,V14 | V12 | V3,0.433564,2.370133,1703.069380,1.590306,0.928537



✓ Processo concluído em: 22m 21.02s
